# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Raja-saab/Flyrank1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

In [3]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

MONTH = "2026-03"

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB connected.")
print("Working month:", MONTH)

DuckDB connected.
Working month: 2026-03


## 1. Unit of analysis + time window

**Unit of analysis:** One row in my modeling feature frame represents **one pseudonymized content item for one calendar month**. I use the daily performance table as the source and aggregate its daily observations to the content-month level.

**Feature window:** For this assignment I use **March 2026** as the development month. Features describe information observed during March and are therefore available at the end of that month.

**Prediction moment:** The decision moment is the end of March 2026.

**Outcome:** The target is whether the content item's Google Search impressions decline by more than 20% in the following month, April 2026.

**Why this grain:** It matches the practical content-refresh decision: at the end of a month, decide which content deserves attention based on what was known up to that point.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 2. Fields: feature / label / context / excluded

**Features — maximum five**

1. `impressions_prev30` — total GSC impressions in the previous 30-day period available at the decision moment.
2. `clicks_prev30` — total GSC clicks in the previous 30-day period.
3. `avg_position` — average GSC search position during the feature window.
4. `days_with_impressions` — number of days in the feature window with at least one GSC impression.
5. `sessions_30d` — total GA4 sessions during the feature window, when GA4 data is available.

**Label / proxy**

`is_declining_next_month` — 1 when April 2026 impressions are more than 20% below March 2026 impressions; otherwise 0. This is the outcome I want to identify, not a feature.

**Context**

`client_hash_id`, `content_hash_id`, and the month/date fields are used for grouping, joining, checking the panel, and possible client-level validation. They are not model features.

**Excluded**

I deliberately exclude `trend_direction`, `trend_pct`, and any April outcome values from the features because they contain or directly describe future/label information. I also exclude pseudonymous IDs because they are identifiers rather than useful content signals.

**Missing values**

GA4 features are only used where `ga4_data_available IS TRUE`. I do not interpret zero-filled GA4 values from unavailable periods as real zero engagement.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification 1 — Grain

I expect the raw warehouse fact table to have one row per `report_date × client × content`. I verify this by searching for duplicate combinations. An empty result means the claimed daily grain holds.


In [6]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date <  DATE '2026-04-01'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate daily-grain combinations found:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate daily-grain combinations found: 0


,report_date,client_hash_id,content_hash_id,row_count


### Verification 2 — March 2026 count and date span

I check how many daily warehouse rows are present in my development month and confirm the earliest and latest dates actually available. This prevents me from assuming that the calendar window is complete when the panel may be unbalanced.


In [7]:
count_window = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date <  DATE '2026-04-01'
""").df()

count_window

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,first_date,last_date,clients,content_items
0,9841378,2026-03-01,2026-03-31,55,331437


### Verification 3 — Search and Analytics availability

I measure how many March rows have usable GSC and GA4 data. I use `IS TRUE` rather than `= TRUE` because the warehouse availability flags can contain NULL values. This makes the availability check explicit and avoids silently treating unknown availability as available.


In [8]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows

    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date <  DATE '2026-04-01'
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


## 4. Data limits

This data can never tell me why a content item's performance changed or prove that a specific content update caused the change. The dataset also has an unbalanced history, so not every client or content item has the same amount of historical data. Some early rows are GSC-only, meaning GA4-based features are unavailable for those observations. Finally, the feature and outcome windows can overlap across adjacent periods, so observations may not be completely independent.

Named limitation: Unbalanced history and incomplete GA4 availability can make comparisons between content items less reliable.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.